In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")

Project root added: C:\Users\USER\Desktop\2026_japan
✓ Imports ready


In [2]:
import joblib
model = joblib.load("../models/lgbm_sapporo_2.pkl")

features = [
    "weekday", "t", "x", "y", "is_weekend",
    "lag_1", "lag_7", "rolling_3", "rolling_7"
]

print("✓ Model loaded")

✓ Model loaded


In [3]:
import pandas as pd

df = pd.read_parquet("../data/processed/sapporo_density.parquet")
is_unknown = (df["x"]==999) & (df["y"]==999)
df = df[~is_unknown].copy() 

df["date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(df["d"], unit="D")
df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

In [4]:
df = df.sort_values(["x","y","t","d"])

df["lag_1"] = df.groupby(["x","y","t"])["count"].shift(1)
df["lag_7"] = df.groupby(["x","y","t"])["count"].shift(7)

df["rolling_3"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(3).mean())
)

df["rolling_7"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df_feat = df.dropna().copy()
print("after feature rows:", len(df_feat))

after feature rows: 4881569


In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb

features = ["weekday","t","x","y","is_weekend","lag_1","lag_7","rolling_3","rolling_7"]
infer_df = df_feat.copy()

infer_df["score"] = model.predict(infer_df[features])
infer_df["score"] = infer_df["score"].clip(lower=0)

df_pred = infer_df[["d","t","x","y","score"]].copy()

In [16]:
import re
from typing import List
from dataclasses import dataclass

SLOTS_PER_DAY = 48
SLOT_MIN = 24*60 // SLOTS_PER_DAY  # 30

WEEKDAY_MAP = {
    "週一":1, "星期一":1, "禮拜一":1,
    "週二":2, "星期二":2, "禮拜二":2,
    "週三":3, "星期三":3, "禮拜三":3,
    "週四":4, "星期四":4, "禮拜四":4,
    "週五":5, "星期五":5, "禮拜五":5,
    "週六":6, "星期六":6, "禮拜六":6,
    "週日":0, "星期日":0, "星期天":0, "禮拜日":0,
}

@dataclass
class CrowdQuery:
    city: str
    date: str | None 
    weekday: int | None      
    hhmm: str | None         # "20:00"
    radius_m: float | None
    place: str | None        # "札幌站" etc.
    place_variants: List[str] | None
    # If you already have lat/lon:
    lat: float | None = None
    lon: float | None = None

def time_to_slot(hhmm: str) -> int:
    h, m = hhmm.split(":")
    minutes = int(h)*60 + int(m)
    return int(round(minutes / SLOT_MIN)) % SLOTS_PER_DAY

def parse_query_rules(text: str, default_city="sapporo") -> CrowdQuery:
    # weekday
    wd = None
    for k,v in WEEKDAY_MAP.items():
        if k in text:
            wd = v
            break

    # time: "20:30"
    m = re.search(r"(\d{1,2}):(\d{2})", text)
    hhmm = None
    if m:
        hhmm = f"{int(m.group(1)):02d}:{int(m.group(2)):02d}"
    else:
        # crude: "晚上8點" / "8點"
        m2 = re.search(r"(晚上|夜晚|下午)?\s*(\d{1,2})\s*點(半)?", text)
        if m2:
            h = int(m2.group(2))
            if m2.group(1) in ("晚上","夜晚") and h < 12:
                h += 12
            if m2.group(1) == "下午" and h < 12:
                h += 12
            minute = 30 if m2.group(3) else 0
            hhmm = f"{h:02d}:{minute:02d}"

    # radius: "附近" -> default; "800m" -> parse
    radius_m = None
    m3 = re.search(r"(\d+(?:\.\d+)?)\s*(m|公尺|米|km|公里)", text.lower())
    if m3:
        val = float(m3.group(1))
        unit = m3.group(2)
        if unit in ("km","公里"):
            val *= 1000
        radius_m = val
    else:
        if "附近" in text:
            radius_m = 800  # default

    # place: very naive (you can do better: NER, dictionary, etc.)
    # Here: take first token before weekday/time keywords
    place = text
    for k in list(WEEKDAY_MAP.keys()) + ["附近"]:
        place = place.replace(k, " ")
    place = place.strip()
    place = place if place else None

    return CrowdQuery(city=default_city, weekday=wd, hhmm=hhmm, radius_m=radius_m, place=place)

In [54]:
from openai import OpenAI
import json
import pandas as pd
import datetime
from dotenv import load_dotenv

load_dotenv()      # 讀 .env 進環境變數
client = OpenAI()

QUERY_SCHEMA = {
  "name": "crowd_query",
  "schema": {
    "type": "object",
    "additionalProperties": False,
    "properties": {
      "city": {"type": "string"},
      "date": {"type": ["string","null"], "description": "YYYY-MM-DD or null"},
      "weekday": {"type": ["integer","null"], "minimum": 0, "maximum": 6},
      "hhmm": {"type": ["string","null"], "pattern": r"^\d{2}:\d{2}$"},
      "radius_m": {"type": ["number","null"], "minimum": 0},
      "place": {"type": ["string","null"]},
      "place_variants": {"type": "array", "items": {"type": "string"}}
    },
    "required": ["city","date","weekday","hhmm","radius_m","place","place_variants"]
  }
}

def parse_query_llm(text: str, default_city="sapporo") -> CrowdQuery:
    TODAY = datetime.date.today()
    prompt = f"""
              你是查詢解析器。今天日期是 {TODAY}。
              把使用者文字轉成 JSON，必須符合 schema。

              規則：
              - weekday 使用 0=週日 ... 6=週六的規律
              - hhmm 一律輸出 24 小時制 "HH:MM"
              - 若有「附近」但沒數字距離，radius_m=800
              - 若沒提城市，city 用 default_city
              - 若提到「今天/明天/後天」，請換算成 date="YYYY-MM-DD"
              - 若 date 有值，weekday 也要填正確（可由 date 推得）
              - place 只能是地點名稱本身（例如「札幌站」），不得包含「週六、07:00、附近」等其他詞；如果使用者沒有輸入地點就回傳 null。
              - place_variants 是使用者所輸入的 place 名稱的別名陣列，產生10個以內的日文、中文或英文別名。例如札幌站的別名有:["札幌站", "札幌車站", "札幌駅", "JR札幌駅", "Sapporo Station", "JR Sapporo Station", "札幌駅 北海道"]
              
              使用者輸入：{text}
              """

    resp = client.responses.create(
        model="gpt-4o",  # 你可換成你要的模型
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": QUERY_SCHEMA["name"],
                "schema": QUERY_SCHEMA["schema"],
                "strict": True
            }
        }
    )
    data = json.loads(resp.output_text)
    print(data)
    return CrowdQuery(**data)

In [8]:
from src.geo.grid_to_latlng import GridLatLngMapper

anchors = [
    {"x": 24, "y": 151, "lat": 43.06918333153887, "lng": 141.35147072116592},  # 札幌站 
    {"x": 24, "y": 148, "lat": 43.07940372979633, "lng": 141.34225589803765},  # 北海道大學
    {"x": 26, "y": 153, "lat": 43.05798589528942, "lng": 141.35402112326315},  # 狸小路商店街
]

mapper = GridLatLngMapper(anchors)

In [ ]:
import numpy as np
import pandas as pd

# scipy 的 KDTree（通常比純 python 快）
from scipy.spatial import KDTree

def build_grid_kdtree(df_pred: pd.DataFrame, mapper):
    # 1) 收集這個城市所有出現過的格子 (x,y)
    grid_xy = df_pred[["x", "y"]].drop_duplicates().reset_index(drop=True)

    # 2) 把每個格子中心轉成 lat/lng
    grid_ll = mapper.transform(grid_xy.copy()).reset_index(drop=True)
    lat = grid_ll["lat"].to_numpy()
    lng = grid_ll["lng"].to_numpy()

    # 3) 建 KDTree：輸入是一堆點 (lat,lng)
    tree = KDTree(np.c_[lat, lng])

    return tree, grid_xy

tree, grid_xy_lookup = build_grid_kdtree(df_pred, mapper)

In [ ]:
import requests

def geocode_place_nominatim(place: str, city_bounds=None, limit=5):
    """
    city_bounds: (lat_min, lng_min, lat_max, lng_max) 可選，用來限制搜尋範圍
    """
    if not place:
        return None

    params = {
        "q": place,
        "format": "json",
        "limit": limit,  # 拿多筆候選，後面挑最合理的
        "namedetails": 1,
        "extratags": 1,
        "accept-language": "zh-TW,zh,ja,en",
        "countrycodes": "jp",
    }

    # 用 viewbox 限制在札幌附近
    if city_bounds is not None:
        lat_min, lng_min, lat_max, lng_max = city_bounds
        params["viewbox"] = f"{lng_min},{lat_max},{lng_max},{lat_min}"
        params["bounded"] = 1

    r = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params=params,
        headers={"User-Agent": "crowd-demo/1.0"}  # 必須要有
    )
    r.raise_for_status()
    data = r.json()
    return data if isinstance(data, list) else []

def pick_best_in_bounds(results, city_bounds):
    if not results:
        return None
    
    lat_min, lng_min, lat_max, lng_max = city_bounds
    
    # 在城市範圍內就加入
    cand = []
    for it in results:
        lat = float(it["lat"]); lng = float(it["lon"])
        if lat_min <= lat <= lat_max and lng_min <= lng <= lng_max:
            cand.append(it)
    
    def importance(it):
        try:
            return float(it.get("importance") or 0.0)
        except Exception:
            return 0.0

    return max(cand, key=importance)


def resolve_place_to_grid(place, tree, grid_xy_lookup, city_bounds=None):
    
    best_overall = None
    best_imp = -1.0

    for v in place:
        results = geocode_place_nominatim(v, city_bounds=city_bounds, limit=5)
        print(results)
        best = pick_best_in_bounds(results, city_bounds)
        if best is None:
            continue

        imp = float(best.get("importance") or 0.0)
        if imp > best_imp:
            best_imp = imp
            best_overall = best

    if best_overall is None:
        return None


    lat = float(best_overall["lat"]); lng = float(best_overall["lon"])
    # KDTree 找最近格子
    _, idx = tree.query([lat, lng], k=1)
    x = int(grid_xy_lookup.loc[idx, "x"])
    y = int(grid_xy_lookup.loc[idx, "y"])
    print("Chosen:", best_overall.get("display_name"),
          "importance=", best_imp, "latlng=", (lat, lng), "grid=", (x, y))
    return (x, y)

In [ ]:
import numpy as np
import sys
from src.util import *

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return float(2*R*np.arcsin(np.sqrt(a)))

def cell_size_m_at(mapper, x0, y0):
    pts = pd.DataFrame([{"x":x0,"y":y0},{"x":x0+1,"y":y0},{"x":x0,"y":y0+1}])
    pts = mapper.transform(pts)
    c = pts.iloc[0]
    px = pts.iloc[1]
    py = pts.iloc[2]
    dx = haversine_m(c.lat, c.lng, px.lat, px.lng)
    dy = haversine_m(c.lat, c.lng, py.lat, py.lng)
    return (dx + dy) / 2

def compute_city_query_bounds(df_grid: pd.DataFrame, mapper):
    """
    df_grid 至少要有 x,y 欄位，代表全市所有可查詢格子。
    回傳 grid bounds + latlng bounds（用四角轉換）
    """
    x_min, x_max = int(df_grid["x"].min()), int(df_grid["x"].max())
    y_min, y_max = int(df_grid["y"].min()), int(df_grid["y"].max())

    corners = pd.DataFrame([
        {"x": x_min, "y": y_min},
        {"x": x_min, "y": y_max},
        {"x": x_max, "y": y_min},
        {"x": x_max, "y": y_max},
    ])
    corners_ll = mapper.transform(corners)

    lat_min, lat_max = float(corners_ll["lat"].min()), float(corners_ll["lat"].max())
    lng_min, lng_max = float(corners_ll["lng"].min()), float(corners_ll["lng"].max())

    return {
        "x_min": x_min, "x_max": x_max,
        "y_min": y_min, "y_max": y_max,
        "lat_min": lat_min, "lat_max": lat_max,
        "lng_min": lng_min, "lng_max": lng_max,
    }

CITY_BOUNDS = compute_city_query_bounds(df_pred[["x","y"]].drop_duplicates(), mapper)


def parse_grid_in_text(text: str):
    m = re.search(r"grid\((\d+)\s*,\s*(\d+)\)", text)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

def answer_query(text: str, df_pred: pd.DataFrame, mapper, coverages=(0.5,0.8,0.95)):
    # 1) parse (先用 rule-based；你要換 LLM 就換這行)
    #q = parse_query_rules(text, default_city="sapporo")
    q = parse_query_llm(text, default_city='sapporo')

    if q.weekday is None or q.hhmm is None:
        return {"error": "缺少星期或時間，請補上例如：週六 20:00"}

    t_slot = time_to_slot(q.hhmm)

    # 2) resolve location -> grid center
    xy = parse_grid_in_text(q.place)
    if xy is None:
        xy = resolve_place_to_grid(q.place_variants, tree, grid_xy_lookup,
                               city_bounds=(CITY_BOUNDS["lat_min"], CITY_BOUNDS["lng_min"],
                                           CITY_BOUNDS["lat_max"], CITY_BOUNDS["lng_max"]))
        if xy is None:
            return {"error": "找不到地點"}
    x0, y0 = xy

    # 3) pick a representative day d for that weekday
    tmp_days = df_pred[["d"]].drop_duplicates().copy()
    tmp_days["weekday"] = (tmp_days["d"] % 7)  
    cand_days = tmp_days[tmp_days["weekday"] == q.weekday]["d"].tolist()
    if not cand_days:
        return {"error": "找不到對應 weekday 的資料日 d，請確認 weekday 定義與 d->weekday 映射一致"}
    d_use = int(cand_days[-1])  # 用最後一個（偏近期）

    # 4) get predicted slice at (d_use, t_slot)
    pred_slice = df_pred[(df_pred["d"]==d_use) & (df_pred["t"]==t_slot)][["x","y","score"]].copy()
    if pred_slice.empty:
        return {"error": "該時段沒有預測資料"}

    # 5) create circles around (x0,y0) within a neighborhood
    cell_m = cell_size_m_at(mapper, x0, y0)
    radius_m = 800 if q.radius_m is None else float(q.radius_m)
    radius_cells = int(np.ceil(radius_m / max(cell_m, 1e-6)))

    MAX_RADIUS_CELLS = 64
    radius_cells = 8 if q.radius_m is None else 8

    cells_xy = pred_slice[["x","y"]].to_numpy()
    dist = np.hypot(cells_xy[:,0]-x0, cells_xy[:,1]-y0)

    # 候選格：先取中心附近一個窗口，避免全圖計算太大
    cand = pred_slice[dist <= radius_cells + 1e-9]
    if len(cand) < 5:
        return {"error": "附近候選格太少，請加大 radius 或檢查格網"}

    cand_xy = cand[["x","y"]].to_numpy()
    p = normalize_nonneg(cand["score"].to_numpy())

    circles = []
    for alpha in coverages:
        win = radius_cells

        while True:
            cand = pred_slice[dist <= win + 1e-9]
            if len(cand) < 5:
                return {"error": "附近候選格太少，請加大 radius 或檢查格網"}

            cand_xy = cand[["x", "y"]].to_numpy()
            p = normalize_nonneg(cand["score"].to_numpy())

            # 固定圓心 (x0,y0)，找最小半徑讓圓內累積機率 >= alpha
            r, idx_circle = circle_radius_by_mass(cand_xy, p, x0, y0, alpha)
            achieved = float(p[idx_circle].sum())  # 實際達到的累積質量（可能 < alpha，若候選窗口太小）

            if achieved >= alpha or win >= MAX_RADIUS_CELLS:
                break

            win *= 2 

        
        circles.append({
            "alpha": alpha,
            "center_grid": {"x": float(x0), "y": float(y0)},  # 固定中心點
            "radius_cells": r,
            "n_cells": int(len(idx_circle)),
            "achieved_mass": achieved,
            "window_cells": win
        })

    return {
        "query": {
            "city": q.city,
            "place": q.place,
            "weekday": q.weekday, "hhmm": q.hhmm, "t_slot": t_slot,
            "d_used": d_use,
            "center_grid": {"x": x0, "y": y0},
            "radius_m": float(radius_m),
            "radius_cells": radius_cells,
        },
        "circles": circles
    }

In [12]:
import folium
from branca.colormap import linear
from branca.element import Template, MacroElement
from folium.plugins import HeatMap

def plot_query_on_map(resp, mapper):
    x0 = resp["query"]["center_grid"]["x"]
    y0 = resp["query"]["center_grid"]["y"]
    d  = resp["query"]["d_used"]
    t  = resp["query"]["t_slot"]

    # grid to lat,lng
    center = mapper.transform(pd.DataFrame([{"x":x0,"y":y0}])).iloc[0]
    lat0, lng0 = float(center.lat), float(center.lng)

    # cell size (meters) around center
    cell_m = cell_size_m_at(mapper, x0, y0)
    cell_area_m2 = cell_m * cell_m
    cell_area_km2 = cell_area_m2 / 1e6

    # 取 95% 那個圈的 window
    c95 = next((c for c in resp["circles"] if abs(c["alpha"] - 0.95) < 1e-9), None)
    win = int(c95.get("window_cells", resp["query"]["radius_cells"])) if c95 else int(resp["query"]["radius_cells"])

    # 取 (d,t) 的預測切片，並限制在窗口內
    sl = df_pred[(df_pred["d"] == d) & (df_pred["t"] == t)][["x","y","score"]].copy()
    dist_grid = np.hypot(sl["x"] - x0, sl["y"] - y0)
    sl = sl[dist_grid <= win + 1e-9].copy()
    if sl.empty:
        raise ValueError("pred_slice is empty for this (d,t).")

    cand_xy = sl[["x","y"]].to_numpy()
    p_pred = normalize_nonneg(sl["score"].to_numpy())

    # 重新用同一批 cand_xy 算 95% 圈（固定中心）
    r95, idx95 = circle_radius_by_mass(cand_xy, p_pred, x0, y0, 0.95)
    achieved95 = float(p_pred[idx95].sum())

    m = folium.Map(location=[lat0, lng0], zoom_start=14)

    # 中心點 marker
    folium.Marker([lat0, lng0], tooltip=f"center grid=({x0},{y0})").add_to(m)

    # 標示網格座標極限範圍（窗口內的 x/y min~max） ---
    '''x_min, x_max = int(sl["x"].min()), int(sl["x"].max())
    y_min, y_max = int(sl["y"].min()), int(sl["y"].max())

    corners = pd.DataFrame([
        {"x": x_min, "y": y_min},
        {"x": x_min, "y": y_max},
        {"x": x_max, "y": y_min},
        {"x": x_max, "y": y_max},
    ])
    corners_ll = mapper.transform(corners)
    lat_min, lat_max = float(corners_ll["lat"].min()), float(corners_ll["lat"].max())
    lng_min, lng_max = float(corners_ll["lng"].min()), float(corners_ll["lng"].max())

    folium.Rectangle(
        bounds=[[lat_min, lng_min], [lat_max, lng_max]],
        color="black",
        weight=2,
        fill=False,
        tooltip=f"GRID BOUNDS (window): x[{x_min},{x_max}] y[{y_min},{y_max}]  |  win={win} cells"
    ).add_to(m)'''
    x_min, x_max = CITY_BOUNDS["x_min"], CITY_BOUNDS["x_max"]
    y_min, y_max = CITY_BOUNDS["y_min"], CITY_BOUNDS["y_max"]
    lat_min, lat_max = CITY_BOUNDS["lat_min"], CITY_BOUNDS["lat_max"]
    lng_min, lng_max = CITY_BOUNDS["lng_min"], CITY_BOUNDS["lng_max"]

    folium.Rectangle(
        bounds=[[lat_min, lng_min], [lat_max, lng_max]],
        color="black",
        weight=2,
        fill=False,
        tooltip=f"CITY QUERY BOUNDS: x[{x_min},{x_max}] y[{y_min},{y_max}]"
    ).add_to(m)

    # 圈圈
    for c in resp["circles"]:

        color = {
            0.5: "red",
            0.8: "orange",
            0.95: "blue"
        }[c["alpha"]]

        radius_m = float(c["radius_cells"] * cell_m)
        folium.Circle(
            location=[lat0, lng0],
            radius=radius_m,
            color=color,
            fill=True,
            fill_opacity=0.10,
            popup=f"alpha={c['alpha']}  r_cells={c['radius_cells']:.2f}  r_m≈{radius_m:.0f}"
        ).add_to(m)

    # 顏色映射（score 越大越紅）
    smin, smax = float(sl["score"].min()), float(sl["score"].max())
    cmap = linear.YlOrRd_09.scale(smin, smax)

    # 把每個格子轉成 lat/lng，並畫密度點；95% 圈內點加粗/更不透明
    ll = mapper.transform(sl[["x","y"]].copy())  # 需回傳含 lat/lng 欄位（依你的 mapper 實作）
    ll = ll.assign(score=sl["score"].values)

    # 計算每點是否在 95% 圈內（grid 距離 <= r95）
    dist_to_center = np.hypot(sl["x"].to_numpy() - x0, sl["y"].to_numpy() - y0)
    in95 = dist_to_center <= r95 + 1e-9

    # 區域密度圖：HeatMap（用 score 當 intensity） ---
    heat_data = [[float(ll.iloc[i].lat), float(ll.iloc[i].lng), float(ll.iloc[i].score)] for i in range(len(ll))]
    HeatMap(
        heat_data,
        name="Density HeatMap (score)",
        min_opacity=0.20,
        radius=22,   # 你可調：越大越糊
        blur=28,     # 你可調：越大越糊
        max_zoom=16
    ).add_to(m)

    for i in range(len(ll)):
        lat, lng = float(ll.iloc[i].lat), float(ll.iloc[i].lng)
        score = float(ll.iloc[i].score)
        color = cmap(score)

        # 點大小
        rr = 2 + 6 * (score - smin) / (smax - smin + 1e-12)

        folium.CircleMarker(
            location=[lat, lng],
            radius=float(rr),
            color="#000000" if in95[i] else "#666666",
            weight=2 if in95[i] else 1,
            fill=True,
            fill_color=color,
            fill_opacity=0.9 if in95[i] else 0.25,
            popup=f"score={score:.3f}  in95={bool(in95[i])}"
        ).add_to(m)


    # 平均密度計算（窗口內 + 95% 圈內） ---
    # 窗口內
    avg_score_win = float(sl["score"].mean())
    avg_density_win_km2 = avg_score_win / (cell_area_km2 + 1e-12)

    # 95% 圈內（用 in95 mask）
    avg_score_95 = float(sl.loc[in95, "score"].mean()) if in95.any() else np.nan
    avg_density_95_km2 = avg_score_95 / (cell_area_km2 + 1e-12) if np.isfinite(avg_score_95) else np.nan

    # 加一個固定在左上角的資訊框
    time_str = f"d={d}, t={t} (approx {t//2:02d}:{(t%2)*30:02d})"
    info_html = f"""
    {{% macro html(this, kwargs) %}}
    <div style="
        position: fixed;
        top: 12px; left: 12px;
        z-index: 9999;
        background: rgba(255,255,255,0.92);
        border: 1px solid #999;
        border-radius: 10px;
        padding: 10px 12px;
        font-size: 12px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        max-width: 320px;
    ">
      <div style="font-weight:700; font-size:13px; margin-bottom:6px;">Crowd / Density Summary</div>
      <div><b>Query</b>: center=({x0},{y0}) | {time_str}</div>
      <div><b>Window</b>: win={win} cells (≈ {win*cell_m:.0f} m)</div>
      <div><b>Grid bounds</b>: x[{x_min},{x_max}], y[{y_min},{y_max}]</div>
      <hr style="margin:8px 0;">
      <div><b>Avg score (window)</b>: {avg_score_win:.3f} /cell</div>
      <div><b>Avg density (window)</b>: {avg_density_win_km2:.1f} /km²</div>
      <div style="margin-top:6px;"><b>95% circle</b>: r95={r95:.2f} cells (≈ {r95*cell_m:.0f} m), achieved={achieved95:.3f}</div>
      <div><b>Avg score (in 95%)</b>: {avg_score_95:.3f} /cell</div>
      <div><b>Avg density (in 95%)</b>: {avg_density_95_km2:.1f} /km²</div>
      <div style="color:#666; margin-top:6px;">
        Note: density assumes each grid cell is ~{cell_m:.0f}m × {cell_m:.0f}m.
      </div>
    </div>
    {{% endmacro %}}
    """
    macro = MacroElement()
    macro._template = Template(info_html)
    m.get_root().add_child(macro)


    # 顏色圖例
    cmap.caption = "predicted score"
    cmap.add_to(m)

    # 圖層控制（可切換 HeatMap/點）
    folium.LayerControl(collapsed=False).add_to(m)

    m.save("prdict_confidence.html")

In [56]:

resp = answer_query("札幌火車站 週六 07:00 附近", df_pred , mapper)
if isinstance(resp, dict) and ("error" in resp or "query" not in resp):
    print("Query failed:", resp)
else:
    plot_query_on_map(resp, mapper)

{'city': 'default_city', 'date': None, 'weekday': 6, 'hhmm': '07:00', 'radius_m': 800, 'place': '札幌火車站', 'place_variants': ['札幌火車站', '札幌站', '札幌車站', '札幌駅', 'JR札幌駅', 'Sapporo Station', 'JR Sapporo Station', '札幌駅 北海道']}
[]
[{'place_id': 251454168, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 1669790050, 'lat': '43.0676395', 'lon': '141.3453006', 'class': 'amenity', 'type': 'restaurant', 'place_rank': 30, 'importance': 7.353398947122939e-05, 'addresstype': 'amenity', 'name': '月見軒 札幌站北口店', 'display_name': '月見軒 札幌站北口店, 3, 北6条西7, 北区, 札幌市, 石狩振興局, 北海道, 北海道地方, 060-0806, 日本', 'extratags': {'cuisine': 'noodle;ramen', 'website': 'http://www.sapporo-honey-angels.com/tukimiken/index.html', 'cuisine:ja': 'ラーメン屋', 'indoor_seating': 'yes', 'outdoor_seating': 'no'}, 'namedetails': {'name': '月見軒', 'name:en': 'Tsukimiken', 'name:ja': '月見軒', 'name:zh': '月見軒 札幌站北口店'}, 'boundingbox': ['43.0675895', '43.0676895', '141.3452506', '141.3453506']